# PySpark — Traitement distribué d'images

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tanouar/Courses/blob/featgen/pySpark/notebooks/04_traitement_images.ipynb)

Notebook conçu pour **Google Colab** — adapté aux débutants en data engineering.

**Ce que tu vas apprendre :**
- Charger des milliers d'images en parallèle avec `spark.read.format("binaryFile")`
- Extraire des métadonnées depuis les noms de fichiers (lazy vs action)
- Écrire une UDF Python (Pillow) exécutée sur les workers Spark
- Visualiser un échantillon d'images directement dans le notebook
- Redimensionner toutes les images de façon distribuée

**Dataset :** Oxford-IIIT Pet Dataset — ~7 300 photos de 37 races de chats et chiens.

In [1]:
## Section 0 — Setup (Colab uniquement)
!pip install -q pyspark findspark pillow matplotlib
!git clone --filter=blob:none --sparse https://github.com/tanouar/1to1code.git -q \
  && cd 1to1code && git sparse-checkout set pySpark -q

import os, sys
PYSPARK_TOOLS = os.path.join('/content', '1to1code', 'pySpark')
if PYSPARK_TOOLS not in sys.path:
    sys.path.insert(0, PYSPARK_TOOLS)

from sparktools import (
    setup_colab,
    download_oxford_pets,
    build_metadata_pipeline,
    add_image_dimensions,
    resize_images,
    show_image_grid,
    show_before_after,
)
from pyspark.sql import functions as F

spark, monitor, helper = setup_colab("PySpark - Traitement Images")

## 1. Télécharger le dataset

Le dataset **Oxford-IIIT Pet** contient ~7 300 photos de chats et chiens (37 races).  
`download_oxford_pets()` télécharge l'archive (~800 MB) et l'extrait dans `/content/oxford_pets/images/`.

> Le téléchargement peut prendre 2-3 minutes selon la connexion Colab.

In [ ]:
# Téléchargement et extraction du dataset
images_path = download_oxford_pets()
print(f"\nImages disponibles dans : {images_path}")

## 2. Chargement distribué — `binaryFile`

`spark.read.format("binaryFile")` parcourt tous les fichiers `.jpg` **en parallèle** sur les workers.  
Chaque image devient une ligne du DataFrame avec ses octets bruts (`content`), son chemin (`path`) et sa taille (`length`).

> **Lazy evaluation :** ce plan d'exécution est créé instantanément — aucune image n'est encore lue.  
> Vérifie dans SparkUI : aucun job ne doit apparaître après cette cellule.

In [ ]:
# Chargement lazy — aucune image lue avant une action
df_images = spark.read.format("binaryFile") \
    .option("pathGlobFilter", "*.jpg") \
    .load(images_path)

df_images.printSchema()
print("\nPlan créé — aucun job visible dans SparkUI (lazy).")

## 3. Qui sont ces images ?

On extrait la race, l'espèce et l'identifiant depuis le **nom de fichier**.  
Format attendu : `Abyssinian_1.jpg` → race `Abyssinian`, espèce `Cat` (première lettre majuscule = chat).

> `build_metadata_pipeline()` est une série de transformations **lazy**.  
> Le premier vrai travail est déclenché par `count()` — c'est à ce moment que Spark lit les noms de fichiers.

In [ ]:
# Extraction des métadonnées depuis les noms de fichiers (lazy)
df_meta = build_metadata_pipeline(df_images)

# Premier count — déclenche la lecture des noms de fichiers
total = monitor.execute_and_monitor(lambda: df_meta.count(), "count images")
print(f"\nTotal : {total} images")

In [ ]:
# Distribution par espèce et race
monitor.execute_and_monitor(
    lambda: df_meta.groupBy("species", "race")
                   .count()
                   .orderBy("species", "count", ascending=False)
                   .show(40, truncate=False),
    "stats par race"
)

## 4. Décoder les images — UDF Pillow

Pour connaître les vraies dimensions `width × height`, il faut **décoder** chaque image.  
`add_image_dimensions()` applique une **UDF** (User Defined Function) :  
une fonction Python ordinaire, ici avec Pillow, mais exécutée sur chaque worker en parallèle.

| Étape | Où ça s'exécute |
|---|---|
| `build_metadata_pipeline()` | Lazy (aucun worker) |
| `add_image_dimensions()` | UDF Pillow sur chaque worker |
| `.count()` ou `.show()` | Déclenche l'exécution distribuée |

> On travaille sur un **échantillon de 300 images** pour rester rapide en Colab.

In [ ]:
# Échantillon de 300 images pour rester rapide
df_sample = df_meta.limit(300)

# Décodage distribué via UDF Pillow — l'action déclenche le travail sur les workers
df_dims = monitor.execute_and_monitor(
    lambda: add_image_dimensions(df_sample).cache(),
    "décodage dimensions (UDF Pillow)"
)
df_dims.count()  # matérialise le cache

print(f"\nColonnes disponibles : {df_dims.columns}")

## 5. Visualiser les images

On collecte **12 images** sur le driver pour les afficher avec matplotlib.  
Les workers ont fait le travail de décodage en parallèle — le driver ne reçoit qu'un petit échantillon.

In [ ]:
# Grille de 12 images avec race et espèce
show_image_grid(df_dims, n=12, title="Échantillon du dataset Oxford-IIIT Pet")

## 6. Dimensions — les images sont-elles toutes de la même taille ?

Dans un pipeline ML, les images doivent souvent être **normalisées** (même taille).  
Ici on explore la distribution des dimensions originales pour comprendre pourquoi un redimensionnement est nécessaire.

In [ ]:
# Statistiques sur les dimensions (min, max, moyenne, écart-type)
monitor.execute_and_monitor(
    lambda: df_dims.select("width", "height", "pixels")
                   .describe()
                   .show(),
    "stats dimensions"
)

In [ ]:
# Orientation des images (paysage / portrait / carré)
monitor.execute_and_monitor(
    lambda: df_dims.groupBy(
        F.when(F.col("width") > F.col("height"), "Paysage")
         .when(F.col("width") < F.col("height"), "Portrait")
         .otherwise("Carré").alias("orientation")
    ).count().orderBy("count", ascending=False).show(),
    "orientation images"
)

## 7. Transformer — redimensionnement distribué

Chaque image a des dimensions différentes (on vient de le voir !).  
Pour qu'un modèle ML puisse les traiter, elles doivent toutes avoir **la même taille**.

`resize_images()` applique une **UDF Pillow** sur chaque worker pour redimensionner à `224×224` px.

> **224×224** est la taille d'entrée standard des CNN pré-entraînés : ResNet, VGG, EfficientNet…  
> Cette transformation est **lazy** — Spark ne redimensionne rien avant `.count()` ou `.show()`.

In [ ]:
# Redimensionnement à 224×224 px — transformation lazy sur df_dims (déjà en cache)
df_resized = resize_images(df_dims, width=224, height=224)

# On matérialise 8 images pour la comparaison visuelle
df_resized_sample = df_resized.limit(8).cache()
monitor.execute_and_monitor(
    lambda: df_resized_sample.count(),
    "redimensionnement échantillon (UDF Pillow)"
)
print("Redimensionnement terminé.")

In [ ]:
# Comparaison visuelle : avant / après redimensionnement (4 images)
show_before_after(df_resized_sample, df_resized_sample, n=4)

## 8. Et le Machine Learning ?

Ce notebook a montré comment PySpark peut **préparer** un dataset d'images à l'échelle :  
téléchargement, chargement distribué, extraction de métadonnées, décodage via UDF, et redimensionnement.

**PySpark n'est pas un framework de Deep Learning**, mais il excelle pour la **préparation des données** :

| Ce que PySpark fait bien | Ce que PyTorch / TensorFlow fait bien |
|---|---|
| Téléchargement & extraction distribués | Entraînement du modèle (GPU) |
| Nettoyage, filtrage, redimensionnement | Transfer Learning (ResNet, EfficientNet…) |
| Export vers GCS / S3 / Azure Blob | Inférence sur de nouvelles images |

**Prochaines étapes possibles :**
- Exporter les images redimensionnées vers un stockage cloud pour les partager avec un cluster GPU
- Utiliser `torch.utils.data.Dataset` pour charger les images préparées par Spark
- Entraîner un modèle de classification sur les 37 races (chats & chiens)